# Лабораторная работа № 2. Сбор информации об ошибках и формирование отчётов об ошибках

## 1. Контекст работы

Вам передан учебный прототип **системы обработки SQL-подобных запросов** к таблице пользователей — тот же, что использовался в предыдущей лабораторной работе. В этот раз исходные ошибки сравнения, хеш-поиска и передачи размера таблицы планировщику исправлены. Алгоритмы поиска (`SeqScan`, `HashTable`, `BST`) считаются рабочими.

Служба поддержки вновь получила **четыре обращения пользователей**. В данном задании неисправности связаны не с реализацией алгоритмов, а с **нарушением согласованности данных, индексов и статистики**. Ваша задача — создать отдельный диагностический модуль, который собирает измеримые показатели (*метрики*), применяет к ним правила и выдаёт заключение по известным типам инцидентов.

**Необходимо отличать наблюдаемый симптом от причины.** Например, недостаточное покрытие таблицы индексом и неправильные индексные ключи — это разные виды нарушения целостности. Аналогично, верное количество строк в статистике ещё не гарантирует правильности распределения значений.

## 2. Как устроена система

### 2.1. Таблица и запросы

Данные хранятся в списке `users`. Одна строка имеет формат `[id, name, age]`, а `columns = ['id', 'name', 'age']` задаёт порядок полей.

Поддерживается только упрощённая форма SQL-подобного запроса:

```sql
select name from users where age >= 18
```

Разрешены операторы `=`, `>`, `<`, `>=`, `<=` и целочисленное значение возраста. Это **не полноценный SQL**: произвольные имена таблиц, `AND`, `OR`, `ORDER BY` не поддерживаются.

### 2.2. Способы поиска

- **SeqScan** — последовательный просмотр записей.
- **HashTable** — хеш-индекс с методом цепочек для поиска по равенству. В индексе хранятся пары «возраст → позиции строк».
- **BST** — бинарное дерево поиска по возрасту, поддерживающее равенство и диапазоны. **Код BST изменять запрещено.**

Индексы ссылаются на позиции строк в `users`. После изменения данных индекс должен соответствовать таблице. Для корректного добавления пользователя предусмотрена `insert_user()`, для изменения возраста — `change_user_age()`: эти операции перестраивают хеш-индекс.

### 2.3. Статистика и планирование

`analyze(table)` сохраняет в `stats` два показателя:

- `analyzed_rows` — количество строк на момент анализа;
- `age_counts` — количество записей для каждого возраста.

Статистика **обновляется только при вызове `analyze()`**, а не автоматически после каждого изменения таблицы. Поэтому расхождение до явного анализа само по себе **не означает ошибку**.

`planner(N, op, value)` выбирает способ поиска по условной стоимости. В модели:

- `cost_SeqScan = N`;
- `cost_Hash = 1 + 4 * 2`;
- `cost_BST = log2(max(N, 1)) * X * 4`, где `X` берётся из `stats['age_counts']`.

### 2.4. Четыре типа известных инцидентов

В качестве базы известных инцидентов рассматриваются следующие категории (их диагностические признаки требуется определить самостоятельно):

| Код | Категория |
|---|---|
| `INC-01` | Индекс не охватывает все строки таблицы |
| `INC-02` | Индексные ключи не соответствуют данным таблицы |
| `INC-03` | Некорректное общее число строк в статистике |
| `INC-04` | Некорректные частоты значений в статистике |

**Название категории — не готовое диагностическое правило.** Определите, какие значения метрик должны подтверждать или опровергать каждый тип проблемы.

## 3. Справочник функций

| Функция или объект | Назначение |
|---|---|
| `users`, `columns` | Таблица пользователей и порядок полей |
| `column_index(column)` | Индекс столбца по имени |
| `compare(a, op, b)` | Проверка условия сравнения |
| `seq_search(table, column, op, value)` | Последовательный поиск |
| `build_hash_index(table, column, bucket_count)` | Создание хеш-индекса по полю |
| `hash_search(table, index, value)` | Поиск с использованием хеш-индекса |
| `BSTNode`, `_bst_insert`, `build_bst_index`, `bst_search` | Дерево поиска; изменять нельзя |
| `analyze(table)` | Явное обновление статистики |
| `stats` | Текущая сохранённая статистика |
| `planner(N, op, value)` | Выбор плана запроса |
| `age_hash_index` | Хеш-индекс по возрасту |
| `insert_user(row)` | Корректное добавление пользователя с перестроением индекса |
| `change_user_age(user_id, new_age)` | Корректное изменение возраста с перестроением индекса |
| `run_query(query_string)` | Выполнение SQL-подобного запроса |
| `reset_scenario(add_new=False)` | Подготовка исходных данных и согласованных метаданных |

## 4. Что разрешено и запрещено изменять

**Разрешено:** добавлять новые ячейки, функции вычисления метрик, диагностические правила, справочник инцидентов, проверки и формирование отчётов. Решение должно работать с существующими объектами системы.

**Запрещено:** изменять алгоритмы `SeqScan`, `HashTable`, `BST`, `analyze`, `planner`, `run_query`; исправлять данные в сценариях ради нужного результата; менять служебные ячейки подготовки неисправностей; использовать номер обращения как готовый диагноз.

**В этой работе исправлять исходные алгоритмы не требуется.** Показанные ниже сбои вводятся намеренно только в тестовых сценариях: таким образом моделируется нарушение состояния системы, которое должен обнаруживать диагностический модуль.

## 5. Порядок выполнения лабораторной

1. Выполните ячейку с реализацией информационной системы и ячейку `reset_scenario()`.
2. Изучите четыре обращения и контрольный сценарий. Каждый запускайте **отдельно**, так как он самостоятельно восстанавливает исходное состояние.
3. Для известных типов неисправностей определите **набор измеряемых метрик**. Необходимо проверить как индекс, так и статистику.
4. Добавьте функции вычисления метрик **без изменения основного кода**.
5. Реализуйте **правила**, которые выявляют отклонения метрик от нормального состояния.
6. Создайте **движок правил**, сопоставляющий комбинации срабатываний с известными инцидентами.
7. Проверьте четыре обращения, исправный контрольный сценарий и, по возможности, одновременное присутствие нескольких неисправностей.
8. Сформируйте диагностический отчёт по каждому обращению: метрики, отклонения, сработавшие правила, возможный инцидент, доказательства.

> **Важно:** для статистики сравнение с текущим состоянием корректно только после явного `analyze()`. В предлагаемых сбойных сценариях он вызывается до нарушения статистических данных.

## 6. Что должно быть в результате

Отчёт по каждому обращению (просто перечислите в комментариях, в текстовой ячейке под обращением или в конце для всех ПЯТИ сценариев):

| № | Измеренные метрики | Какие значения не в норме? | Сработавшие правила | Возможные инциденты и обоснование |
|---|---|---|---|---|
| 1 | | | | |


---

## 7. Исходный код системы

Следующая ячейка содержит корректную реализацию. Ниже расположены служебная подготовка и обращения пользователей. Диагностический модуль необходимо написать после них.


## Примечание

Работа продолжает лабораторную № 1: структура системы и типы запросов знакомы, однако теперь код поиска исправен. **Нужно не искать ошибки в алгоритмах, а разработать диагностический модуль**, который по состоянию системы обнаруживает и классифицирует нарушения.

In [ ]:
users = [[1, 'Anna', 18],
        [2, 'Ivan', 21],
        [3, 'Petr', 17],
        [4, 'Maria', 25],
        [5, 'Alex', 18],
        [6, 'Olga', 30],
        [7, 'Nikita', 21]]

columns = ['id', 'name', 'age']


def column_index(column):
    return columns.index(column)


def compare(a, op, b):
    if op == '=':
        return a == b
    if op == '>':
        return a > b
    if op == '<':
        return a < b
    if op == '>=':
        return a >= b
    if op == '<=':
        return a <= b
    raise ValueError(f'Неизвестный оператор: {op}')


#
# Блок с реализацией последовательного поиска
#

def seq_search(table, column, op, value):
    c = column_index(column)
    result = []

    for row in table:
        if compare(row[c], op, value):
            result.append(row)

    return result


#
#  Блок с реализацией хэширования
#
def build_hash_index(table, column, bucket_count=30):
    c = column_index(column)
    buckets = [[] for _ in range(bucket_count)]

    for row_id, row in enumerate(table):
        key = row[c]
        bucket_id = hash(key) % bucket_count
        bucket = buckets[bucket_id]

        found = False
        for item in bucket:
            if item[0] == key:
                item[1].append(row_id)
                found = True
                break

        if not found:
            bucket.append([key, [row_id]])

    return buckets


def hash_search(table, index, value):
    bucket_count = len(index)
    bucket_id = hash(value) % bucket_count
    bucket = index[bucket_id]

    for key, row_ids in bucket:
        if key == value:
            return [table[row_id] for row_id in row_ids]

    return []



#
# Блок с реализацией BST
#

class BSTNode:
    def __init__(self, key, row_id):
        self.key = key
        self.row_ids = [row_id]
        self.left = None
        self.right = None


def _bst_insert(node, key, row_id):
    if node is None:
        return BSTNode(key, row_id)

    if key < node.key:
        node.left = _bst_insert(node.left, key, row_id)
    elif key > node.key:
        node.right = _bst_insert(node.right, key, row_id)
    else:
        node.row_ids.append(row_id)

    return node


def build_bst_index(table, column):
    c = column_index(column)
    root = None

    for row_id, row in enumerate(table):
        root = _bst_insert(root, row[c], row_id)

    return root


def bst_search(table, root, op, value):
    row_ids = []

    def walk(node):
        if node is None:
            return

        if op == '=':
            if value < node.key:
                walk(node.left)
            elif value > node.key:
                walk(node.right)
            else:
                row_ids.extend(node.row_ids)
            return

        if op == '>':
            if node.key > value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.right)
            return

        if op == '>=':
            if node.key >= value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.right)
            return

        if op == '<':
            if node.key < value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.left)
            return

        if op == '<=':
            if node.key <= value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.left)
            return

        raise ValueError(f'Неизвестный оператор: {op}')

    walk(root)
    row_ids.sort()
    return [table[row_id] for row_id in row_ids]


#
# Блок с планировщиком и анализом
#
import math

stats = {}

def analyze(table):
    counts = {}
    for row in table:
        age = row[column_index('age')]
        counts[age] = counts.get(age, 0) + 1
    stats['age_counts'] = counts
    stats['analyzed_rows'] = len(table)
    print('Статистика обновлена:', stats)


def planner(N, op, value):
    if N <= 0:
        return [0, 'SeqScan']
    x = stats.get('age_counts', {}).get(value, 0)
    cost_SeqScan = N
    cost_Hash = 1 + 4 * 2
    cost_BST = math.log2(max(N, 1)) * x * 4
    if op != '=':
        if cost_BST <= cost_SeqScan:
            return [cost_BST, 'BST']
        return [cost_SeqScan, 'SeqScan']
    if cost_Hash <= cost_SeqScan:
        return [cost_Hash, 'HashTable']
    return [cost_SeqScan, 'SeqScan']
#
#
#


age_hash_index = build_hash_index(users, 'age', bucket_count=5)

def run_query(query_string):
    query = query_string.lower().strip().split()

    if (len(query) != 8 or query[:4] != ['select', 'name', 'from', 'users']) or (query[4] != 'where') or (query[5] != 'age') or (query[6] not in ('=', '>', '<', '>=', '<=')):
        raise ValueError('Ожидается: select name from users where age = 18')

    value = int(query[7])
    op = query[6]
    plan = planner(len(users), op, value)

    if plan[1] == 'SeqScan':
        rows = seq_search(users, 'age', op, value)
    elif plan[1] == 'HashTable':
        rows = hash_search(users, age_hash_index, value)
    else:
        idx = build_bst_index(users, 'age')
        rows = bst_search(users, idx, op, value)

    names = [r[column_index('name')] for r in rows]
    print(f'Запрос: {query_string}')
    print(f'План: {plan[1]}, стоимость: {plan[0]:.3f}, X={stats.get("age_counts", {}).get(value, 0)}, N={len(users)}')
    print('Результат:', names)
    return names


# Поддерживаемые операции изменения данных.
# Обновление таблицы через эти функции сохраняет согласованность индекса.
# Статистика, как и раньше, обновляется отдельно через analyze().

def insert_user(row):
    global age_hash_index

    if any(existing[0] == row[0] for existing in users):
        raise ValueError('Пользователь с таким id уже существует')

    users.append(list(row))
    age_hash_index = build_hash_index(users, 'age', bucket_count=5)


def change_user_age(user_id, new_age):

    global age_hash_index

    for row in users:
        if row[0] == user_id:
            row[column_index('age')] = new_age
            age_hash_index = build_hash_index(users, 'age', bucket_count=5)
            return
    raise ValueError('Пользователь не найден')


print('ЭТАП 1. Проверка исправной системы')
analyze(users)
run_query('select name from users where age = 18')

print('\nЭТАП 2. Корректно добавляем пользователя')
insert_user([8, 'New_1', 18])
analyze(users)
run_query('select name from users where age = 18')


ЭТАП 1. Проверка исправной системы
Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Запрос: select name from users where age = 18
План: SeqScan, стоимость: 7.000, X=2, N=7
Результат: ['Anna', 'Alex']

ЭТАП 2. Корректно добавляем пользователя
Статистика обновлена: {'age_counts': {18: 3, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 8}
Запрос: select name from users where age = 18
План: SeqScan, стоимость: 8.000, X=3, N=8
Результат: ['Anna', 'Alex', 'New_1']


['Anna', 'Alex', 'New_1']

## Обращения пользователей: тестовые сценарии

Запускайте сценарии **по отдельности**, предварительно выполнив код системы и служебную ячейку ниже. Каждый сценарий начинает работу с `reset_scenario()` и создаёт определённое состояние данных.

**Не исправляйте сценарии и не запускайте `analyze()` или перестроение индекса поверх неисправности до измерения метрик:** это изменит исследуемое состояние. Для перехода к следующему сценарию просто запустите его ячейку.

In [ ]:
INITIAL_USERS = [
    [1, 'Anna', 18], [2, 'Ivan', 21], [3, 'Petr', 17],
    [4, 'Maria', 25], [5, 'Alex', 18], [6, 'Olga', 30],
    [7, 'Nikita', 21],
]

#Возвращает таблицу, индекс и статистику в изначальное состояние.
def reset_scenario(add_new=False):
    global users, age_hash_index

    users = [row.copy() for row in INITIAL_USERS]
    age_hash_index = build_hash_index(users, 'age', bucket_count=5)

    if add_new:
        for i in range(15):
            insert_user([8 + i, f'New_{i + 1}', 18])

    stats.clear()
    analyze(users)


print('Сценарии готовы к запуску.')


Сценарии готовы к запуску.




---



### Обращение № 1. После массового импорта поиск возвращает не все записи

> «Мы загрузили ещё 15 пользователей, всем указали возраст 18 лет. В общем списке они есть, а поиск по возрасту показывает только часть людей. Можете проверить, откуда расхождение?»

Сценарий моделирует загрузку данных **в обход штатной операции `insert_user()`**. После загрузки статистика явно обновляется. Не редактируйте код сценария — исследуйте состояние таблицы и индекса.

In [ ]:
reset_scenario()

for i in range(15):
    users.append([8 + i, f'New_{i + 1}', 18])
analyze(users)

print('Фактических записей с age=18:', sum(row[2] == 18 for row in users))
run_query('select name from users where age = 18')

#diagnose()

Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Фактических записей с age=18: 17
Запрос: select name from users where age = 18
План: HashTable, стоимость: 9.000, X=17, N=22
Результат: ['Anna', 'Alex']


['Anna', 'Alex']



---



### Обращение № 2. После изменения возраста выдача противоречит карточке пользователя

> «Пользователю Maria недавно исправили возраст с 25 на 26. В карточке теперь 26, но поиск по возрасту 25 всё ещё показывает Maria, а по возрасту 26 — нет. Проверьте, пожалуйста».

Сценарий моделирует изменение поля `age` напрямую в данных, без штатной операции `change_user_age()`. Статистика после изменения корректно пересчитывается.

In [ ]:
reset_scenario(add_new=True)


users[3][column_index('age')] = 26
analyze(users)

print('Карточка Maria:', users[3], '\n\n')
run_query('select name from users where age = 25')
print('\n\n')
run_query('select name from users where age = 26')

Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 26: 1, 30: 1}, 'analyzed_rows': 22}
Карточка Maria: [4, 'Maria', 26] 


Запрос: select name from users where age = 25
План: HashTable, стоимость: 9.000, X=0, N=22
Результат: ['Maria']



Запрос: select name from users where age = 26
План: HashTable, стоимость: 9.000, X=1, N=22
Результат: []


[]



---



### Обращение № 3. Статистика показывает неправильный размер таблицы

> «Мы выполнили обновление статистики. В таблице 22 записи, но в сведениях о последнем анализе указано меньше строк. При этом пользователи в таблице никуда не исчезали».

Сценарий моделирует повреждение **одного служебного показателя** после успешного `analyze()`. Проверяйте статистику и реальные данные независимо.

In [ ]:
reset_scenario(add_new=True)

stats['analyzed_rows'] -= 5

print('Строк в таблице:', len(users))
print('Количество строк по статистике:', stats['analyzed_rows'])
print('Сумма частот по возрастам:', sum(stats['age_counts'].values()))
run_query('select name from users where age = 18')

Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Строк в таблице: 22
Количество строк по статистике: 17
Сумма частот по возрастам: 22
Запрос: select name from users where age = 18
План: HashTable, стоимость: 9.000, X=17, N=22
Результат: ['Anna', 'Alex', 'New_1', 'New_2', 'New_3', 'New_4', 'New_5', 'New_6', 'New_7', 'New_8', 'New_9', 'New_10', 'New_11', 'New_12', 'New_13', 'New_14', 'New_15']


['Anna',
 'Alex',
 'New_1',
 'New_2',
 'New_3',
 'New_4',
 'New_5',
 'New_6',
 'New_7',
 'New_8',
 'New_9',
 'New_10',
 'New_11',
 'New_12',
 'New_13',
 'New_14',
 'New_15']



---



### Обращение № 4. Распределение возрастов в статистике выглядит неверным

> «После обновления статистики общее число пользователей совпало с таблицей, но распределение по возрастам вызывает вопросы. В отчёте пользователей 18 лет стало меньше, а 21 года — больше, хотя карточки пользователей не менялись».

Сценарий моделирует повреждение **распределения значений** после успешного `analyze()`. Число строк `analyzed_rows` остаётся правильным.

In [ ]:
reset_scenario(add_new=True)

stats['age_counts'][18] -= 4
stats['age_counts'][21] += 4

print('Строк в таблице:', len(users))
print('По статистике всего строк:', stats['analyzed_rows'])
print('По статистике возраст 18:', stats['age_counts'][18])
print('В таблице возраст 18:', sum(row[2] == 18 for row in users))
print('По статистике возраст 21:', stats['age_counts'][21])
print('В таблице возраст 21:', sum(row[2] == 21 for row in users))

Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Строк в таблице: 22
По статистике всего строк: 22
По статистике возраст 18: 13
В таблице возраст 18: 17
По статистике возраст 21: 6
В таблице возраст 21: 2




---



### Обращение № 5. Контрольная проверка после корректного обслуживания

> «Пользователей загрузили обычным способом, статистику обновили. Жалоб на поисковую выдачу нет, но хотелось бы убедиться, что индекс и статистика согласованы».

В этом сценарии **не предусмотрено неисправностей**. Диагностический движок должен сообщить, что отклонения по заданным проверкам не обнаружены.

In [ ]:
reset_scenario(add_new=True)
print('Число строк:', len(users))
run_query('select name from users where age = 18')
print('Статистика:', stats)

Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Число строк: 22
Запрос: select name from users where age = 18
План: HashTable, стоимость: 9.000, X=17, N=22
Результат: ['Anna', 'Alex', 'New_1', 'New_2', 'New_3', 'New_4', 'New_5', 'New_6', 'New_7', 'New_8', 'New_9', 'New_10', 'New_11', 'New_12', 'New_13', 'New_14', 'New_15']
Статистика: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}




---



---



---



## 8. Разработка диагностического модуля

В следующих ячейках реализуйте **две части задания**:

**Часть A. Сбор метрик.** Получите числовые показатели состояния индекса и статистики. Метрики должны работать на всех сценариях, в том числе исправном. Убедитесь, что количество записей в индексе само по себе не гарантирует правильности ключей.

**Часть B. Система правил.** Реализуйте проверку условий по метрикам, сопоставление с четырьмя известными категориями инцидентов и вывод результатов.

Можно использовать предложенный каркас или написать функции с другим интерфейсом. Сохраняйте исходные кодовые ячейки без изменений.

In [ ]:
# Известные категории инцидентов — НЕ готовые правила
KNOWN_INCIDENTS = {
    'INC-01': 'Индекс охватывает не все строки таблицы',
    'INC-02': 'Индексные ключи не соответствуют данным таблицы',
    'INC-03': 'Неправильное общее количество строк в статистике',
    'INC-04': 'Неправильное распределение значений в статистике',
}

# Ваше решение должно заключаться в изменении кода ниже

def collect_metrics(table, index, statistics):
    # Здесь вам предстоит организовать сбор нужных метрик
    raise NotImplementedError('Реализуйте сбор метрик')


def evaluate_rules(metrics):
    # Здесь вам предстоит определить границы метрик(правила), после которых мы можем считать, что значения вышли из нормы
    raise NotImplementedError('Реализуйте проверку правил')


def match_incidents(triggered_rules):
    #Здесь нужно сопоставить правила и значения статистики, после чего определить, какие из предложенных инцедентов могут сработать
    raise NotImplementedError('Реализуйте распознавание инцидентов')


def diagnose():
    metrics = collect_metrics(users, age_hash_index, stats)
    fired = evaluate_rules(metrics)
    matches = match_incidents(fired)
    result = {
        'metrics': metrics,
        'triggered_rules': fired,
        'possible_incidents': matches,
    }
    print('Метрики:', result['metrics'])
    print('Сработавшие правила:', result['triggered_rules'])
    print('Возможные инциденты:', result['possible_incidents'] or 'Отклонений не найдено')
    return result


reset_scenario()

for i in range(15):
    users.append([8 + i, f'New_{i + 1}', 18])
analyze(users)

print('Фактических записей с age=18:', sum(row[2] == 18 for row in users))
run_query('select name from users where age = 18')

diagnose()



---



## 9. Проверка и отчёт

После реализации функций выполните **по очереди** ячейки обращений № 1–5, а вслед за каждой — `diagnose()`. Покажите рассчитанные метрики, сработавшие правила и найденные инциденты.

Для расширенного испытания создайте самостоятельно сценарий, в котором **две неисправности присутствуют одновременно**, и убедитесь, что движок сообщает обе. Дополнительный контроль: измените данные через `insert_user()` или `change_user_age()` с последующим `analyze()` — в исправном состоянии предупреждений быть не должно.


In [ ]:
# После запуска любого обращения используйте эту ячейку:
#diagnose()

# Запускайте вызов только после завершения реализации функций выше.